# Lecture 7 — Class Exercise
## Heatmap & Waterfall: Netflix Catalogue

> **Push to:** `week07/lecture07_exercise.ipynb`

**Rules:**
1. Heatmap: colour scale must match the data type (sequential for counts, diverging for above/below)
2. Waterfall: use green for additions, red for subtractions, blue for totals
3. Insight title tells the setup-conflict-resolution story (or at minimum states the finding)
4. Annotate at least one cell or bar directly

---


In [2]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

df = pd.read_csv('/content/netflix_catalogue.csv')
print(f"Loaded: {len(df)} titles")
print(df['type'].value_counts())
print(df.head())


Loaded: 3000 titles
type
Movie      1974
TV Show    1026
Name: count, dtype: int64
      type  release_year  added_year             genre        country rating  \
0    Movie          2014        2016  Sci-Fi & Fantasy         France  PG-13   
1    Movie          2010        2014     Documentaries  United States  TV-MA   
2  TV Show          2011        2012     Kids & Family  United States  TV-14   
3    Movie          2016        2018             Anime          India     PG   
4    Movie          2014        2016     Kids & Family         Canada  TV-MA   

   duration  
0       157  
1       127  
2         6  
3       134  
4        77  


In [3]:
print("Genres:", df['genre'].value_counts().head(8))
print("\nCountries:", df['country'].value_counts().head(8))
print("\nRatings:", df['rating'].value_counts())


Genres: genre
Sports                244
Sci-Fi & Fantasy      213
Kids & Family         209
Crime                 206
Drama                 204
Horror                199
Action & Adventure    198
Thrillers             195
Name: count, dtype: int64

Countries: country
United States     932
India             337
United Kingdom    261
Japan             187
France            176
Canada            164
South Korea       151
Mexico            138
Name: count, dtype: int64

Ratings: rating
TV-MA    840
TV-14    733
PG-13    589
R        312
PG       196
TV-PG    128
G         92
TV-Y7     57
TV-G      53
Name: count, dtype: int64


## Task 1 — Heatmap: content by rating and release decade

**What to build:** A heatmap showing the number of titles by **content rating** (y-axis) and **decade** (x-axis).

**Requirements:**
- Create a 'decade' column: `df['decade'] = (df['release_year'] // 10 * 10).astype(str) + 's'`
- Filter to TV-14, TV-MA, PG-13, R, PG only (most common ratings)
- Sequential colour scale (Blues)
- Values shown in cells (`text_auto=True`)
- Insight title about which rating dominates which decade


In [5]:
# Task 1
# YOUR CODE HERE
import pandas as pd
import plotly.express as px

# Create decade column
df['decade'] = (df['release_year'] // 10 * 10).astype(str) + 's'

# Filter ratings
ratings_filter = ['TV-14', 'TV-MA', 'PG-13', 'R', 'PG']
filtered_df = df[df['rating'].isin(ratings_filter)]

# Create grouped data
heatmap_data = (
    filtered_df
    .groupby(['rating', 'decade'])
    .size()
    .reset_index(name='count')
)

# Pivot table for heatmap
pivot_df = heatmap_data.pivot(
    index='rating',
    columns='decade',
    values='count'
).fillna(0)

# Find peak value location
max_value = pivot_df.values.max()

peak_position = (
    pivot_df.stack()
    .idxmax()
)

peak_rating = peak_position[0]
peak_decade = peak_position[1]

# Create heatmap
fig = px.imshow(
    pivot_df,
    text_auto=True,
    color_continuous_scale='Blues',
    aspect='auto',
    labels=dict(
        x='Release Decade',
        y='Content Rating',
        color='Number of Titles'
    ),
    title='TV-14 and TV-MA Titles Dominate Streaming Content Across Recent Decades'
)

# Add annotation for peak value
fig.add_annotation(
    x=peak_decade,
    y=peak_rating,
    text=f'🔥 Peak: {int(max_value)}',
    showarrow=True,
    arrowhead=3,
    arrowsize=1.5,
    arrowwidth=2,
    arrowcolor='red',
    font=dict(size=13, color='red'),
    bgcolor='white',
    bordercolor='red',
    borderwidth=1
)

# Improve layout
fig.update_layout(
    xaxis_title='Decade',
    yaxis_title='Rating',
    template='plotly_white'
)

fig.show()

## Task 2 — Waterfall: Movie vs TV Show additions by year

**What to build:** A waterfall chart showing how Netflix's **Movie library** grew year by year (2015-2022).

**Requirements:**
- Filter to Movies only
- Group by `added_year`, count titles per year
- Final bar should be the cumulative total
- Green bars (additions), blue total
- Annotation on the year with the largest single addition
- Insight title naming the growth story


In [7]:
# Task 2
# YOUR CODE HERE
import pandas as pd
import plotly.graph_objects as go

# Load dataset
df = pd.read_csv("netflix_catalogue.csv")

# Filter Movies only and years 2015–2022
movies_df = df[
    (df['type'] == 'Movie') &
    (df['added_year'].between(2015, 2022))
]

# Group by added_year
yearly_additions = (
    movies_df
    .groupby('added_year')
    .size()
    .reset_index(name='count')
    .sort_values('added_year')
)

# Find year with largest addition
peak_row = yearly_additions.loc[yearly_additions['count'].idxmax()]
peak_year = int(peak_row['added_year'])
peak_value = int(peak_row['count'])

# Prepare waterfall data
x_vals = yearly_additions['added_year'].astype(str).tolist() + ['Total']
y_vals = yearly_additions['count'].tolist() + [yearly_additions['count'].sum()]
measure_vals = ['relative'] * len(yearly_additions) + ['total']

# Create Waterfall Chart
fig = go.Figure(go.Waterfall(
    name="Movie Growth",
    orientation="v",
    measure=measure_vals,
    x=x_vals,
    y=y_vals,
    text=[str(v) for v in y_vals],
    textposition="outside",

    increasing=dict(
        marker=dict(color='green')
    ),

    totals=dict(
        marker=dict(color='blue')
    )
))

# Annotate peak year
fig.add_annotation(
    x=str(peak_year),
    y=peak_value,
    text=f'🔥 Highest Growth<br>{peak_value} Movies',
    showarrow=True,
    arrowhead=3,
    arrowsize=1.3,
    arrowwidth=2,
    arrowcolor='red',
    font=dict(size=12, color='red'),
    bgcolor='white',
    bordercolor='red'
)

# Update layout
fig.update_layout(
    title='Netflix Movie Library Saw Its Biggest Expansion During the Streaming Boom Years',
    xaxis_title='Year Added',
    yaxis_title='Number of Movies Added',
    template='plotly_white',
    waterfallgap=0.3
)

fig.show()